# Train the ScreenRun cursor detector on Colab

Select **Runtime → Change runtime type → GPU** first. Locally run `bash training/create_colab_bundle.sh`, then upload the resulting `yolo-cursor-colab.zip` when prompted below. The bundle contains compressed source screenshots and cursor sprites, not generated training images.

In [ ]:
# Confirm the assigned accelerator before spending time uploading data.
import subprocess, torch
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.is_available(), 'Enable a GPU runtime and reconnect.'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name}; VRAM: {props.total_memory / 1024**3:.1f} GiB')

In [ ]:
%pip install -q ultralytics==8.4.115 polars==1.35.2 pyarrow onnx onnxslim

In [ ]:
from google.colab import files
from pathlib import Path
import os, shutil, zipfile

uploaded = files.upload()
bundle = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
assert bundle, 'Upload yolo-cursor-colab.zip.'
workspace = Path('/content/yolo-cursor')
if workspace.exists():
    shutil.rmtree(workspace)
with zipfile.ZipFile(bundle) as archive:
    archive.extractall('/content')
os.chdir(workspace)
print('Workspace:', Path.cwd())

In [ ]:
# Preflight: decode actual runtime samples and inspect dimensions/labels.
from collections import Counter
from training.runtime_compositor import RuntimeCursorDataset

preview = RuntimeCursorDataset(split='train', length=256, negative_rate=0.05, dynamic=False)
samples = [preview[i] for i in range(len(preview))]
assert all(sample['image'].size == (640, 640) for sample in samples)
boxes = [sample['bbox'] for sample in samples if sample['bbox'] is not None]
assert boxes and all(0 < value <= 1 for box in boxes for value in box)
print('Samples:', len(samples), 'positive:', len(boxes), 'negative:', len(samples) - len(boxes))
print('Cursor mix:', Counter(sample['cursor_type'] for sample in samples))
print('Box pixel widths:', round(min(b[2] for b in boxes) * 640), 'to', round(max(b[2] for b in boxes) * 640))
print('Box pixel heights:', round(min(b[3] for b in boxes) * 640), 'to', round(max(b[3] for b in boxes) * 640))

In [ ]:
# CUDA training. Ultralytics prints loss, VRAM, throughput and validation metrics live.
import os
vram_gib = props.total_memory / 1024**3
batch = 128 if vram_gib >= 38 else 64 if vram_gib >= 22 else 32
print(f'Using batch {batch} for {vram_gib:.1f} GiB VRAM')
os.environ.update({
    'CURSOR_DEVICE': '0',
    'CURSOR_BATCH': str(batch),
    'CURSOR_WORKERS': '4',
    'CURSOR_TRAIN_SAMPLES': '10000',
    'CURSOR_VAL_SAMPLES': '1000',
    'CURSOR_NEGATIVE_RATE': '0.05',
    'CURSOR_FROZEN_EPOCHS': '10',
    'CURSOR_UNFROZEN_EPOCHS': '15',
})
%run training/train_augmented.py

In [ ]:
# Show the recorded per-epoch metrics.
import pandas as pd
from IPython.display import display
for csv_path in sorted(Path('training/runs').glob('*/results.csv')):
    print(csv_path)
    display(pd.read_csv(csv_path).tail())

In [ ]:
# Export the best final checkpoint to fixed-size ONNX for the WebGPU demo.
from ultralytics import YOLO
from google.colab import files

best = Path('training/runs/yolo26n-cursor-runtime-unfrozen/weights/best.pt')
assert best.is_file(), f'Missing checkpoint: {best}'
onnx_path = Path(YOLO(best).export(format='onnx', imgsz=640, opset=17, simplify=True, dynamic=False))
print('Exported:', onnx_path, f'({onnx_path.stat().st_size / 1024**2:.1f} MiB)')
files.download(str(onnx_path))